# 🗄️ Step 3: Load Cleaned Data into MySQL (`03_load_to_mysql.ipynb`)

### Overview of MySQL Loading Steps
1. Reads MySQL credentials from environment variables (`.env` file via `load_dotenv()`) and connects to MySQL using `pymysql.connect()` and `connect.cursor()`.
2. Executes `CREATE DATABASE IF NOT EXISTS earthquake_db` and commits.
3. Uses SQLAlchemy `create_engine()` and `df.to_sql("earthquakes", con=engine, if_exists="replace", index=False, chunksize=10000)` to bulk-load all `104,464` cleaned records.
4. Verifies the row count and previews the first 5 rows directly from MySQL using `cur.execute()` and `cur.fetchall()`.

In [1]:
import os
import pandas as pd
import pymysql
from sqlalchemy import create_engine
from dotenv import load_dotenv

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

INPUT_PATH = "data/earthquakes_cleaned.csv"
df = pd.read_csv(INPUT_PATH)
df["time"] = pd.to_datetime(df["time"])
df["updated"] = pd.to_datetime(df["updated"])
print(f"Loaded {len(df)} cleaned rows ready for MySQL.")

Loaded 104464 cleaned rows ready for MySQL.


### 3.1 Create Database `earthquake_db` Using `pymysql` Cursor

In [2]:
# Read MySQL credentials from environment variables (.env file, never pushed to GitHub)
load_dotenv()

HOST = os.getenv("DB_HOST", "localhost")
PORT = int(os.getenv("DB_PORT", "3306"))
USER = os.getenv("DB_USER", "root")
PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME", "earthquake_db")

if PASSWORD is None:
    raise ValueError("DB_PASSWORD is not set. Copy .env.example to .env and add your MySQL password.")

connect = pymysql.connect(
    host=HOST,
    port=PORT,
    user=USER,
    password=PASSWORD,
)
cur = connect.cursor()
cur.execute(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}")
connect.commit()
cur.close()
connect.close()
print(f"Database '{DB_NAME}' is ready.")

Database 'earthquake_db' is ready.


### 3.2 Bulk-Insert DataFrame into `earthquakes` Table Using SQLAlchemy `to_sql()`

In [3]:
connection_string = f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}"
engine = create_engine(connection_string)

print("Inserting rows into MySQL table 'earthquakes'...")
df.to_sql(
    name="earthquakes",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=10000,
)
print("Successfully loaded table 'earthquakes'!")

Inserting rows into MySQL table 'earthquakes'...
Successfully loaded table 'earthquakes'!


### 3.3 Verify Row Count & Sample Records from MySQL (`cur.fetchall()`)

In [4]:
connect = pymysql.connect(
    host=HOST,
    port=PORT,
    user=USER,
    password=PASSWORD,
    database=DB_NAME,
)
cur = connect.cursor()

cur.execute("SELECT COUNT(*) FROM earthquakes")
row_count = cur.fetchall()[0][0]
print("Total rows verified in MySQL 'earthquakes' table:", row_count)

cur.execute("SELECT id, time, country, mag, depth_km, severity_flag FROM earthquakes ORDER BY mag DESC LIMIT 5")
top5 = cur.fetchall()
cols = [desc[0] for desc in cur.description]
cur.close()
connect.close()

pd.DataFrame(top5, columns=cols)

Total rows verified in MySQL 'earthquakes' table: 104464
                 id                     time  ...  depth_km severity_flag
92506    us6000qw60  2025-07-29 23:24:52.483  ...     35.00   destructive
11087  ak0219neiszm  2021-07-29 06:15:49.188  ...     35.00   destructive
5691     us7000dflf  2021-03-04 19:28:33.178  ...     28.93   destructive
14459    us6000f53e  2021-08-12 18:35:17.231  ...     22.79   destructive
45509    us6000jllz  2023-02-06 01:17:34.342  ...     10.00   destructive

[5 rows x 6 columns]
